# REUTERS NEWS AGENT — DDG + EVALUATOR-OPTIMIZER

**Versión:** `v0.13-explained`  
**Última modificación:** `2026-09-11 20:36 CEST`  
**Rama:** `fix/chapter2-news-agent`  
**Modelo:** `gpt-4.1-mini`

Esta versión mantiene el comportamiento de `v0.12`, pero añade **explicaciones y comentarios didácticos** para entender mejor cómo funciona el patrón **Evaluator-Optimizer**.

## Idea general

El flujo es:

```text
Pregunta del usuario
      │
      ▼
Searcher (gpt-4.1-mini)
      │
      ├── usa search_reuters()
      │         │
      │         ▼
      │      DuckDuckGo
      │         │
      │         ▼
      │   URLs Reuters reales
      │
      ▼
Respuesta candidata
      │
      ▼
Validación determinista en Python
      │
      ▼
Evaluator (gpt-4.1-mini)
      │
      ├── successful ──► STOP
      │
      └── unsuccessful
              │
              ▼
       feedback al Searcher
              │
              └── nueva iteración
```

La idea importante es separar tres responsabilidades:

1. **Retrieval:** encontrar artículos reales de Reuters.
2. **Generación:** construir una respuesta legible con esos resultados.
3. **Evaluación:** comprobar si la respuesta cumple exactamente lo pedido.


## 1. Instalar dependencias

- `openai`: cliente oficial de OpenAI.
- `openai-agents`: SDK de agentes que aporta `Agent`, `Runner` y herramientas.
- `ddgs`: buscador que usamos para encontrar URLs Reuters porque el `web_search` alojado no estaba recuperando Reuters correctamente.


In [ ]:
!pip install -U openai openai-agents ddgs -q


## 2. Imports, API key y ventana temporal

Aquí cargamos las librerías y definimos la **ventana temporal permitida**.

En este laboratorio usamos:

```python
TODAY = datetime.now().date()
START_DATE = TODAY - timedelta(days=2)
```

Por tanto, solo aceptaremos artículos cuya fecha esté entre `START_DATE` y `TODAY`, ambos incluidos.


In [ ]:
import json, os, re, sys, importlib.metadata as im
from dataclasses import dataclass
from datetime import datetime, timedelta, date
from typing import Literal
from urllib.parse import urlsplit, urlunsplit

from ddgs import DDGS
from google.colab import userdata
from agents import Agent, Runner
from agents.decorators import tool

# Recuperamos la API key guardada como secreto en Google Colab.
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Ventana de fechas que consideramos válida.
TODAY = datetime.now().date()
START_DATE = TODAY - timedelta(days=2)

# Información útil para saber exactamente qué entorno estamos ejecutando.
print('Python:', sys.version)
print('openai:', im.version('openai'))
print('openai-agents:', im.version('openai-agents'))
print('ddgs:', im.version('ddgs'))
print('API key presente:', bool(os.environ.get('OPENAI_API_KEY')))
print('Ventana:', START_DATE.isoformat(), '->', TODAY.isoformat())


## 3. Helpers de validación Reuters

Estas funciones **no dependen del LLM**. Son comprobaciones deterministas en Python.

Eso es importante porque un LLM puede equivocarse al decir que una URL es válida o que una fecha entra en el rango. Python, en cambio, aplica reglas exactas.

### Qué hace cada helper

- `normalize_reuters_url()`: acepta únicamente dominios `reuters.com` y normaliza la URL.
- `date_from_reuters_url()`: extrae la fecha `YYYY-MM-DD` que Reuters suele incluir en la URL.
- `in_required_window()`: comprueba que la fecha esté en la ventana permitida.
- `extract_reuters_urls()`: extrae las URLs Reuters que aparezcan en la respuesta del agente.
- `requested_count()`: intenta averiguar cuántos artículos ha pedido el usuario. Si no lo dice, usa 5.


In [ ]:
# Reuters suele incluir la fecha del artículo en la propia URL.
DATE_RE = re.compile(r'(20\d{2}-\d{2}-\d{2})')

# Expresión regular sencilla para encontrar URLs dentro de una respuesta de texto.
URL_RE = re.compile(r'https?://[^\s)\]>]+')

def normalize_reuters_url(url: str) -> str | None:
    """Devuelve una URL Reuters normalizada o None si no pertenece a Reuters."""
    try:
        p = urlsplit(url)
    except Exception:
        return None

    host = (p.hostname or '').lower().rstrip('.')

    # Solo aceptamos Reuters y sus subdominios.
    if not (host == 'reuters.com' or host.endswith('.reuters.com')):
        return None

    if p.scheme not in {'http', 'https'}:
        return None

    path = p.path.rstrip('/')
    if not path:
        return None

    # Eliminamos parámetros y fragmentos para comparar URLs de forma consistente.
    return urlunsplit(('https', host, path + '/', '', ''))


def date_from_reuters_url(url: str) -> date | None:
    """Extrae la fecha YYYY-MM-DD que aparece en la ruta de una URL Reuters."""
    m = DATE_RE.search(urlsplit(url).path)
    if not m:
        return None

    try:
        return date.fromisoformat(m.group(1))
    except ValueError:
        return None


def in_required_window(url: str) -> bool:
    """True si la URL Reuters contiene una fecha dentro de la ventana permitida."""
    d = date_from_reuters_url(url)
    return d is not None and START_DATE <= d <= TODAY


def extract_reuters_urls(text: str) -> list[str]:
    """Extrae URLs Reuters distintas de una respuesta generada por el agente."""
    out, seen = [], set()

    for raw in URL_RE.findall(text):
        raw = raw.rstrip('.,;')
        u = normalize_reuters_url(raw)

        if u and u not in seen:
            seen.add(u)
            out.append(u)

    return out


def requested_count(text: str, default: int = 5) -> int:
    """Intenta detectar cuántos artículos solicita el usuario.

    Ejemplos:
      "latest 5 articles" -> 5
      "3 Reuters articles" -> 3

    Si no aparece un número explícito, usamos 5.
    """
    patterns = [
        r'\blatest\s+(\d+)\b',
        r'\b(\d+)\s+(?:Reuters\s+)?articles?\b',
        r'\b(\d+)\s+(?:news|items?)\b',
    ]

    for pat in patterns:
        m = re.search(pat, text, re.I)
        if m:
            return int(m.group(1))

    return default


## 4. Function tool: búsqueda Reuters con DDG

Aquí está una de las piezas principales.

El decorador:

```python
@tool
```

convierte una función Python normal en una **herramienta que el agente puede decidir utilizar**.

El LLM no ejecuta DuckDuckGo directamente. Lo que hace es pedir algo parecido a:

```text
search_reuters("NVIDIA earnings")
```

Entonces Python ejecuta la función, DDG devuelve resultados y esos resultados vuelven al agente como observación.

Esto es un ejemplo claro del patrón:

```text
LLM decide acción → Tool ejecuta → LLM recibe observación
```

La herramienta filtra además los resultados antes de entregarlos al agente: solo Reuters, URLs válidas y fechas dentro de la ventana.


In [ ]:
@tool
def search_reuters(query: str, max_results: int = 10) -> str:
    """Busca artículos Reuters utilizando DDG.

    Args:
        query: Palabras clave del tema/empresa. El agente NO debe incluir
               site:reuters.com porque la función lo añade automáticamente.
        max_results: Número máximo de resultados DDG que queremos inspeccionar.

    Returns:
        Un JSON en texto con los artículos Reuters válidos encontrados.
        Ese texto vuelve al LLM como resultado de la herramienta.
    """

    # Restringimos la búsqueda a Reuters desde la propia query de DDG.
    search_query = f'site:reuters.com {query}'.strip()
    print('\n[DDG TOOL] query:', search_query)

    try:
        raw = DDGS().text(
            search_query,
            region='us-en',
            safesearch='off',
            timelimit='w',
            # Evitamos valores absurdamente altos aunque el agente los pida.
            max_results=max(1, min(max_results, 20)),
        )
    except Exception as e:
        # Si DDG falla, devolvemos un error estructurado al agente.
        msg = {
            'error': f'{type(e).__name__}: {e}',
            'query': search_query,
            'results': [],
        }
        print('[DDG TOOL] ERROR:', msg['error'])
        return json.dumps(msg, ensure_ascii=False)

    results, seen = [], set()

    for item in raw or []:
        href = item.get('href') or item.get('url') or ''

        # Comprobación 1: la URL debe ser realmente Reuters.
        url = normalize_reuters_url(href)
        if not url or url in seen:
            continue

        # Comprobación 2: la URL debe contener una fecha válida.
        article_date = date_from_reuters_url(url)

        # Comprobación 3: esa fecha debe estar dentro del rango solicitado.
        if article_date is None or not (START_DATE <= article_date <= TODAY):
            continue

        seen.add(url)

        # Solo devolvemos al agente información procedente del buscador:
        # título, URL, fecha y snippet.
        results.append({
            'title': item.get('title') or '',
            'url': url,
            'publication_date': article_date.isoformat(),
            'snippet': item.get('body') or item.get('snippet') or '',
        })

    # Estas trazas permiten ver en Colab qué encontró realmente la herramienta.
    print(f'[DDG TOOL] valid Reuters results in window: {len(results)}')
    for r in results:
        print('  ', r['publication_date'], '-', r['title'])
        print('     ', r['url'])

    # JSON mantiene claramente separados los campos para el LLM.
    return json.dumps({
        'query': search_query,
        'window_start': START_DATE.isoformat(),
        'window_end': TODAY.isoformat(),
        'results': results,
    }, ensure_ascii=False)


## 5. Searcher y Evaluator

Aquí creamos **dos agentes con responsabilidades diferentes**.

### `web_news_searcher`

Es el agente que intenta resolver la petición. Puede llamar a `search_reuters()` una o varias veces y después redacta una respuesta.

### `news_evaluator`

No busca noticias. Su trabajo es **juzgar la respuesta del Searcher**.

Devuelve una estructura:

```python
EvaluationFeedback(
    feedback="...",
    score="successful" | "unsuccessful"
)
```

Esta separación es precisamente el patrón **Evaluator-Optimizer**:

```text
Generator/Searcher → Evaluator → feedback → Generator/Searcher
```

El `Literal[...]` es importante: obliga al resultado estructurado a usar solo uno de los dos estados permitidos.


In [ ]:
SEARCHER_INSTRUCTIONS = f"""
You are a financial-news research agent.

Reuters discovery MUST be done with the local search_reuters tool.
Do not use memory as evidence. Do not invent headlines, dates, summaries, or URLs.

The allowed publication window is {START_DATE.isoformat()} through {TODAY.isoformat()}, inclusive.

Only direct reuters.com URLs returned by the tool are valid.
Copy URLs exactly from tool output.

If the first tool call is insufficient, call the tool again with different topic/company keywords.
Search requested companies/topics separately when useful.
Prefer newest items and remove duplicates.

Market quotes, ETF/index pages and generic stock-price pages do not count.

For every final item provide:
- headline
- publication date
- short summary based only on title/snippet
- Publisher Reuters
- direct Reuters URL

Return the exact number requested if enough verified tool results exist.
Otherwise return only verified items.
"""

# Agente generador/optimizador:
# busca información y redacta la respuesta candidata.
web_news_searcher = Agent(
    name='web_news_searcher',
    model='gpt-4.1-mini',
    instructions=SEARCHER_INSTRUCTIONS,
    tools=[search_reuters],
)


@dataclass
class EvaluationFeedback:
    # Feedback textual que volverá al Searcher si la respuesta falla.
    feedback: str

    # Solo permitimos estos dos valores.
    score: Literal['successful', 'unsuccessful']


EVALUATOR_INSTRUCTIONS = f"""
You are a strict evaluator of a Reuters financial-news result.

The allowed date window is {START_DATE.isoformat()} through {TODAY.isoformat()}, inclusive.

You receive:
- the original request
- the generated answer
- the requested article count
- the valid Reuters URL count
- the valid Reuters URLs

Successful only if the answer contains at least the requested number
of distinct genuine Reuters URLs, all inside the date window, and each
item has headline/date/summary/Publisher Reuters/direct URL and is
relevant to the original request.

Zero results or too few results is always unsuccessful.
Never declare success merely because the answer honestly says no results were found.

The score field MUST be exactly one of these two strings:
"successful" or "unsuccessful".

Return concise actionable feedback.
"""

# Agente evaluador:
# no tiene herramientas porque no necesita buscar; solo evalúa lo recibido.
news_evaluator = Agent(
    name='news_evaluator',
    model='gpt-4.1-mini',
    instructions=EVALUATOR_INSTRUCTIONS,
    output_type=EvaluationFeedback,
)


## 6. Bucle Evaluator-Optimizer

Esta es la parte central del laboratorio.

En cada iteración:

1. El Searcher genera una respuesta.
2. Python extrae y valida las URLs Reuters.
3. El Evaluator juzga la respuesta.
4. Si falla, su `feedback` se añade al siguiente prompt.
5. El Searcher prueba una estrategia distinta.
6. Si aprueba, hacemos `break` y terminamos.

Además guardamos la mejor respuesta encontrada con `best_answer`. Esto evita que una iteración posterior peor destruya una respuesta anterior mejor.


In [ ]:
async def main() -> None:
    # ---------------------------------------------------------------
    # 1) Leer petición del usuario y determinar cuántos artículos pide.
    # ---------------------------------------------------------------
    msg = input("User's request: " ).strip()
    target_count = requested_count(msg, default=5)

    print('\nRequested count:', target_count)
    print('Date window:', START_DATE.isoformat(), '->', TODAY.isoformat())

    # Máximo número de ciclos Searcher -> Evaluator.
    max_iterations = 4

    # El feedback empieza vacío. Solo existe después de un intento fallido.
    feedback = None

    # Última respuesta producida por el Searcher.
    latest_answer = ''

    # ---------------------------------------------------------------
    # 2) Guardamos la mejor respuesta aunque nunca lleguemos a aprobar.
    # ---------------------------------------------------------------
    best_answer = ''
    best_valid_count = -1

    # Si el Evaluator aprueba una respuesta, se guarda aquí.
    approved_answer = None

    # ---------------------------------------------------------------
    # 3) Bucle Evaluator-Optimizer.
    # ---------------------------------------------------------------
    for iteration in range(1, max_iterations + 1):
        print(
            '\n\033[92m'
            + f'************************** NEWS SEARCH {iteration} **************************'
            + '\033[0m'
        )

        # Primer intento: enviamos solo la pregunta original.
        if feedback is None:
            search_prompt = msg

        # Siguientes intentos: añadimos el feedback del Evaluator.
        # Esto es lo que permite al "Optimizer" corregir su estrategia.
        else:
            search_prompt = (
                f"{msg}\n\n"
                "The previous answer failed evaluation.\n"
                f"Evaluator feedback: {feedback}\n\n"
                "Run NEW Reuters searches with the search_reuters tool. "
                "Use different company/topic keywords where necessary. "
                "Do not reuse unsupported claims."
            )

        # -----------------------------------------------------------
        # 4) SEARCHER: genera una nueva respuesta usando la herramienta.
        # -----------------------------------------------------------
        search_result = await Runner.run(web_news_searcher, search_prompt)
        latest_answer = str(search_result.final_output)

        print('\nGENERATED ANSWER:\n')
        print(latest_answer)

        # -----------------------------------------------------------
        # 5) VALIDACIÓN DETERMINISTA EN PYTHON.
        #
        # No confiamos solo en que el LLM diga que las URLs son Reuters:
        # las extraemos y verificamos nosotros.
        # -----------------------------------------------------------
        urls = extract_reuters_urls(latest_answer)
        valid_urls = [u for u in urls if in_required_window(u)]

        print(
            '\n\033[93m'
            '************************** DETERMINISTIC VALIDATION **************************'
            '\033[0m'
        )
        print('Reuters URLs in answer:', len(urls))
        print('Valid Reuters URLs in date window:', len(valid_urls))

        for u in valid_urls:
            print('  VALID:', u)

        # -----------------------------------------------------------
        # 6) Recordar la mejor respuesta vista hasta ahora.
        # -----------------------------------------------------------
        if len(valid_urls) > best_valid_count:
            best_valid_count = len(valid_urls)
            best_answer = latest_answer

        # -----------------------------------------------------------
        # 7) Construir el input del Evaluator.
        # -----------------------------------------------------------
        evaluator_input = (
            f"ORIGINAL REQUEST:\n{msg}\n\n"
            f"GENERATED ANSWER:\n{latest_answer}\n\n"
            f"REQUESTED COUNT: {target_count}\n"
            f"VALID REUTERS URL COUNT: {len(valid_urls)}\n"
            "VALID REUTERS URLS:\n"
            + ('\n'.join(valid_urls) if valid_urls else 'NONE')
        )

        print(
            '\n\033[92m'
            '************************** RUNNING EVALUATION **************************'
            '\033[0m'
        )

        # -----------------------------------------------------------
        # 8) EVALUATOR: devuelve score + feedback.
        # -----------------------------------------------------------
        eval_result = await Runner.run(news_evaluator, evaluator_input)
        result: EvaluationFeedback = eval_result.final_output

        # -----------------------------------------------------------
        # 9) Barrera determinista.
        #
        # Aunque el Evaluator se equivocara, Python impide aprobar si
        # hay menos URLs válidas que las solicitadas.
        # -----------------------------------------------------------
        if len(valid_urls) < target_count:
            result.score = 'unsuccessful'
            result.feedback = (
                f'Only {len(valid_urls)} valid Reuters URLs were present; '
                f'{target_count} are required. '
                'Search again with new Reuters queries.'
            )

        print('Evaluator score:', result.score)
        print('Evaluator feedback:', result.feedback)

        # -----------------------------------------------------------
        # 10) Condición de éxito.
        #
        # Deben cumplirse LAS DOS condiciones:
        #   - evaluator == successful
        #   - URLs válidas >= número solicitado
        # -----------------------------------------------------------
        if result.score == 'successful' and len(valid_urls) >= target_count:
            approved_answer = latest_answer
            best_answer = latest_answer
            best_valid_count = len(valid_urls)

            print('Evaluation successful ==> stopping iteration.')
            break

        # Si no aprobó, el feedback pasa a la siguiente iteración.
        feedback = result.feedback

        if iteration == max_iterations:
            print('Reached max_iterations ==> stopping iteration.')

    # ---------------------------------------------------------------
    # 11) Elegir respuesta final.
    #
    # Prioridad:
    #   1. respuesta aprobada
    #   2. mejor respuesta encontrada
    #   3. última respuesta como último recurso
    # ---------------------------------------------------------------
    final_answer = approved_answer or best_answer or latest_answer

    print(
        '\n\033[92m'
        '************************** FINAL NEWS SET **************************'
        '\033[0m'
    )

    if approved_answer is not None:
        print(
            f'Using APPROVED answer with '
            f'{best_valid_count} valid Reuters URLs.\n'
        )
    else:
        print(
            f'No answer was fully approved; returning BEST candidate with '
            f'{best_valid_count} valid Reuters URLs.\n'
        )

    print(final_answer)


## 7. Ejecutar

En Colab usamos:

```python
await main()
```

porque el notebook ya ejecuta dentro de un event loop. En un script Python normal sería habitual usar `asyncio.run(...)`, pero en Jupyter/Colab `await` es la opción más sencilla.

### Qué deberías observar

En una ejecución típica puedes ver algo así:

```text
NEWS SEARCH 1
    ↓
0 resultados válidos
    ↓
Evaluator: unsuccessful
    ↓
feedback

NEWS SEARCH 2
    ↓
el agente prueba varias queries
    ↓
5 Reuters válidas
    ↓
Evaluator: successful
    ↓
STOP
```

Ese comportamiento muestra el **Evaluator-Optimizer** funcionando: el segundo intento cambia su estrategia a partir del feedback del primero.


In [ ]:
await main()
